<a href="https://colab.research.google.com/github/DhruvGangwar320/GitHUB/blob/main/ml_feature_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [5]:
import pandas as pd

data = pd.DataFrame({
    'StudyHours': [2,4,6,4,5,6,7,7,4,5,1,1,6,9,5,3,5,5,8,6],
    'Attendance': [65,70,75,74,85,50,94,93,52,64,74,58,42,94,72,90,80,69,53,83],
    'SleepHours': [6,6,6,7,7,6,6,6,7,7,7,8,8,9,3,8,9,6,4,7],
    'PreviousScore': [55,60,65,70,100,100,100,55,45,60,60,74,85,23,43,43,80,50,77,90],
})

data['FinalScore'] = (
    data['Attendance']*0.3+ data['StudyHours']*5 +data['PreviousScore']*0.5 +5)

print(data.head())

   StudyHours  Attendance  SleepHours  PreviousScore  FinalScore
0           2          65           6             55        62.0
1           4          70           6             60        76.0
2           6          75           6             65        90.0
3           4          74           7             70        82.2
4           5          85           7            100       105.5


In [6]:
# Correlation with target
correlation = data.corr()  # .corr() is a function in pandas,Compares every column with every other column
print("Correlation with Target:")
print(correlation['FinalScore'])
# output will be a matrix as it compares every colum with every other out of which we select final score column.

Correlation with Target:
StudyHours       0.758046
Attendance       0.307778
SleepHours      -0.053946
PreviousScore    0.659603
FinalScore       1.000000
Name: FinalScore, dtype: float64


In [7]:
from sklearn.feature_selection import chi2

# Convert to classification (Pass/Fail)
data['Result'] = (data['FinalScore'] > data['FinalScore'].mean()).astype(int)

print(data)



    StudyHours  Attendance  SleepHours  PreviousScore  FinalScore  Result
0            2          65           6             55        62.0       0
1            4          70           6             60        76.0       0
2            6          75           6             65        90.0       1
3            4          74           7             70        82.2       0
4            5          85           7            100       105.5       1
5            6          50           6            100       100.0       1
6            7          94           6            100       118.2       1
7            7          93           6             55        95.4       1
8            4          52           7             45        63.1       0
9            5          64           7             60        79.2       0
10           1          74           7             60        62.2       0
11           1          58           8             74        64.4       0
12           6          42           8

In [8]:
X = data[['StudyHours', 'Attendance', 'SleepHours', 'PreviousScore']]
y = data['Result']

chi_scores, p_values = chi2(X, y)


chi_df = pd.DataFrame({
    'Feature': X.columns,
    'Chi2 Score': chi_scores,
    'p-value': p_values
})
chi_df['p-value'] = chi_df['p-value'].apply(lambda x: format(x, '.9f'))
print("\nChi-Square Results:\n")
print(chi_df.sort_values(by='Chi2 Score', ascending=False))

#Higher Chi-score → more important feature
#p-value < 0.05 → significant (important)
#p-value > 0.05 → not reliable/



Chi-Square Results:

         Feature  Chi2 Score      p-value
3  PreviousScore   34.625468  0.000000004
0     StudyHours    9.707071  0.001835604
1     Attendance    2.589422  0.107579560
2     SleepHours    0.067669  0.794761191


In [11]:
def backward_elimination(X, y, threshold=0.05): #If p-value > 0.05 → feature is not important
    X = sm.add_constant(X)

    while True:
        model = sm.OLS(y, X).fit()
        p_values = model.pvalues
        #Tells how important each feature is, Smaller p-value = more important feature
        max_p = p_values.max()
        if max_p > threshold:
            remove_feature = p_values.idxmax() #idxmax() → gives feature name with highest p-value
            X = X.drop(columns=[remove_feature])
            print("Removed:", remove_feature)
        else:
            break

    return X.columns

print("\nBackward Elimination:")
selected_features = backward_elimination(X, y)
print("Final Features:", selected_features)


Backward Elimination:
Removed: SleepHours
Final Features: Index(['const', 'StudyHours', 'Attendance', 'PreviousScore'], dtype='object')


In [12]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

X = data[['StudyHours', 'Attendance', 'SleepHours', 'PreviousScore']]
y = data['FinalScore']

# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lasso = Lasso(alpha=0.1) #Alpha decides how much we punish large coefficients
lasso.fit(X_scaled, y)

lasso_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lasso.coef_
})

print("\nLasso Results:\n")
print(lasso_df)


Lasso Results:

         Feature  Coefficient
0     StudyHours    10.236977
1     Attendance     4.430699
2     SleepHours     0.000000
3  PreviousScore    10.344250
